Let me get the data on dengue from [Peru](https://www.datosabiertos.gob.pe/dataset/vigilancia-epidemiol%C3%B3gica-de-dengue):

In [1]:
import pandas as pd

linkData="https://github.com/MagallanesAtAlacip/datafiles/raw/main/dengue_peru.csv"

dengue = pd.read_csv(linkData)

# checking format
dengue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501236 entries, 0 to 501235
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   departamento  501236 non-null  object
 1   provincia     501236 non-null  object
 2   distrito      501236 non-null  object
 3   ano           501236 non-null  int64 
 4   semana        501236 non-null  int64 
 5   sexo          501236 non-null  object
 6   edad          501236 non-null  int64 
 7   enfermedad    501236 non-null  object
 8   case          501236 non-null  int64 
 9   edad_grupos   501236 non-null  object
dtypes: int64(4), object(6)
memory usage: 38.2+ MB


In [2]:
# Each row is a person:
dengue.head()

,departamento,provincia,distrito,ano,semana,sexo,edad,enfermedad,case,edad_grupos
0,HUANUCO,LEONCIO PRADO,LUYANDO,2000,47,M,9,1_SIN_SEÑALES,1,a_menor_a_16
1,HUANUCO,LEONCIO PRADO,LUYANDO,2000,40,F,18,1_SIN_SEÑALES,1,b_entre_16y50
2,HUANUCO,LEONCIO PRADO,JOSE CRESPO Y CASTILLO,2000,48,F,32,1_SIN_SEÑALES,1,b_entre_16y50
3,HUANUCO,LEONCIO PRADO,JOSE CRESPO Y CASTILLO,2000,37,F,40,1_SIN_SEÑALES,1,b_entre_16y50
4,HUANUCO,LEONCIO PRADO,MARIANO DAMASO BERAUN,2000,42,M,16,1_SIN_SEÑALES,1,b_entre_16y50


In [3]:
# exploring
dengue.enfermedad.value_counts()

,count
enfermedad,
1_SIN_SEÑALES,443996
2_ALARMA,54981
3_GRAVE,2259


In [4]:
# exploring
dengue.ano.describe()

,ano
count,501236.000000
mean,2014.772129
std,6.146461
min,2000.000000
25%,2011.000000
50%,2016.000000
75%,2020.000000
max,2022.000000


In [5]:
# exploring
dengue['edad_grupos'].value_counts(sort=False)

,count
edad_grupos,
a_menor_a_16,137072
b_entre_16y50,295018
c_mayor_a_50,69146


# Prepare to plot

In [6]:
# !pip install "vegafusion[embed]>=1.5.0"
# !pip install "vl-convert-python>=1.6.0"

In [7]:
import altair as alt
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

# Univariate

In [8]:
# Create aggregated data
all_depts = dengue['departamento'].value_counts().reset_index()
all_depts.columns = ['departamento', 'count']
##
all_depts

,departamento,count
0,PIURA,126400
1,LORETO,92496
2,UCAYALI,43707
3,MADRE DE DIOS,30562
4,TUMBES,27313
5,SAN MARTIN,26759
6,LA LIBERTAD,24377
7,ICA,21324
8,CAJAMARCA,19409
9,JUNIN,19199


In [9]:
bars = alt.Chart(all_depts).mark_bar(
    color='steelblue').encode(
    y=alt.Y('departamento:N', sort='-x', title='Department'),
    x=alt.X('count:Q', title='Number of Cases'),
    tooltip=['departamento', 'count']
)

bars

alt.Chart(...)

In [10]:
# Sort descending
labels = alt.Chart(all_depts).mark_text(
    dx=20,
    fontSize=10
).encode(
    y=alt.Y('departamento:N', sort='-x'),
    x=alt.X('count:Q'),
    text=alt.Text('count:Q')
)

(bars + labels).properties(
    title='All Departments - Total Dengue Cases (Sorted)',
    width=600,
    height=700
)#.interactive()

alt.LayerChart(...)

# Bivariate

In [11]:

# Age distribution by ALL departments
age_dept = dengue.groupby(['departamento', 'edad_grupos']).size().reset_index()
age_dept.columns = ['departamento', 'edad_grupos', 'count']

age_dept


,departamento,edad_grupos,count
0,AMAZONAS,a_menor_a_16,3379
1,AMAZONAS,b_entre_16y50,7298
2,AMAZONAS,c_mayor_a_50,1340
3,ANCASH,a_menor_a_16,1557
4,ANCASH,b_entre_16y50,4483
...,...,...,...
57,TUMBES,b_entre_16y50,16459
58,TUMBES,c_mayor_a_50,4016
59,UCAYALI,a_menor_a_16,15623
60,UCAYALI,b_entre_16y50,23342


In [12]:

alt.Chart(age_dept).mark_bar(
    size=15
).encode(
    x=alt.X('departamento:N',
            sort='-y',  # Sort by total cases descending
            title='Department'),
    y=alt.Y('count:Q', title='Number of Cases'),
    color=alt.Color('edad_grupos:N',
                    scale=alt.Scale(
                        domain=['a_menor_a_16', 'b_entre_16y50', 'c_mayor_a_50'],
                        range=['#1f77b4', '#ff7f0e', '#2ca02c']
                    ),
                    title='Age Group'),
    tooltip=['departamento', 'edad_grupos', 'count'],
    xOffset='edad_grupos:N'
).properties(
    title='Age Distribution by All Departments (Grouped)',
    width=800,
    height=500
).interactive()

alt.Chart(...)

In [13]:
alt.Chart(age_dept).mark_rect().encode(
    x=alt.X('departamento:N',
            sort='-y',
            title='Department'),
    y=alt.Y('edad_grupos:N',
            sort=['a_menor_a_16', 'b_entre_16y50', 'c_mayor_a_50'],
            title='Age Group'),
    color=alt.Color('count:Q',
                    scale=alt.Scale(scheme='viridis'),
                    title='Cases'),
    tooltip=['departamento', 'edad_grupos', 'count']
).properties(
    title='Heatmap: Age Groups by All Departments',
    width=800,
    height=200
).interactive()

alt.Chart(...)

In [14]:

alt.Chart(age_dept).mark_bar().encode(
    y=alt.Y('departamento:N',
            sort='-x',
            title='Department'),
    x=alt.X('count:Q', title='Number of Cases'),
    color=alt.Color('edad_grupos:N',
                    scale=alt.Scale(
                        domain=['a_menor_a_16', 'b_entre_16y50', 'c_mayor_a_50'],
                        range=['#1f77b4', '#ff7f0e', '#2ca02c']
                    ),
                    title='Age Group'),
    tooltip=['departamento', 'edad_grupos', 'count'],
    xOffset='edad_grupos:N'
).properties(
    title='Age Distribution by Department',
    width=700,
    height=800
).interactive()

alt.Chart(...)

In [15]:

# Faceted bar chart by age group
alt.Chart(age_dept).mark_bar().encode(
    y=alt.Y('departamento:N',
            sort='-x',
            title='Department'),
    x=alt.X('count:Q',
            title='Number of Cases'),  # Log scale for better visualization

    tooltip=['departamento', 'edad_grupos', 'count']
).properties(
    title='Age Distribution by Department',
    width=200,
    height=800
).facet(
    column='edad_grupos:N'
).interactive()

alt.FacetChart(...)

# Mining location

Let's use _departamento_ and _provincia_:

In [16]:
# --- Step 1: total cases per (year, departamento, provincia, enfermedad severity) ---
indexList = ['ano', 'departamento', 'provincia', 'enfermedad']
aggregator = {'case': ['sum']}
ByYearPlace = dengue.groupby(indexList, observed=True).agg(aggregator)

# --- Step 2: pivot enfermedad categories into columns ---
# fillna(0) is essential: if a province-year had zero cases of a given
# severity (e.g. no 2_ALARMA that year), unstack() leaves NaN there.
# Without filling, shareAlarma below would be NaN instead of 0.
ByYearPlace_wide = ByYearPlace.unstack().fillna(0)

# --- Step 3: share of ALARMA cases out of all cases, per province-year ---
sumCases = ByYearPlace_wide.sum(axis=1)
shareAlarma = ByYearPlace_wide.loc[:, ('case', 'sum', '2_ALARMA')] / sumCases
shareAlarma.name = 'shareAlarma'
shareAlarma = shareAlarma.reset_index()

# --- Step 4: for each (year, departamento), find the provincia with the
# highest ALARMA share -- i.e. the "worst" province that year ---
# NOTE: ties (two provinces sharing the same max) are broken arbitrarily
# by idxmax (first occurrence wins). Check with:
# (shareAlarma.groupby(['ano','departamento'])['shareAlarma']
#     .apply(lambda s: (s == s.max()).sum()) > 1).sum()
where = shareAlarma.groupby(['ano', 'departamento'])['shareAlarma'].idxmax()
worst_prov_year = shareAlarma.loc[where].reset_index(drop=True)

# --- Step 5: keep only cases where the "worst" province actually had
# some ALARMA share (excludes departamento-years where no province
# had any alarma cases at all, so the max would just be 0) ---
worst_ProvYear_alarma = (
    worst_prov_year[worst_prov_year.shareAlarma > 0]
    .loc[:, ['departamento', 'provincia']]
    .reset_index(drop=True)
)

# --- Step 6: count how many years each provincia was its departamento's
# "worst" (highest ALARMA share) province ---
indexList = ['departamento', 'provincia']
aggregator = {'provincia': ['count']}
worst_ProvYear_alarma_Frequency = (
    worst_ProvYear_alarma.groupby(indexList, observed=True).agg(aggregator)
)

# --- Step 7: clean up column name and keep only provinces affected > 2 years ---
worst_ProvYear_alarma_Frequency.columns = ['yearsAffected']
worst_ProvYear_alarma_Frequency = worst_ProvYear_alarma_Frequency[
    worst_ProvYear_alarma_Frequency.yearsAffected > 2
]
worst_ProvYear_alarma_Frequency.reset_index(inplace=True)
worst_ProvYear_alarma_Frequency

,departamento,provincia,yearsAffected
0,AMAZONAS,BAGUA,3
1,AMAZONAS,CHACHAPOYAS,3
2,AMAZONAS,UTCUBAMBA,5
3,ANCASH,CASMA,3
4,ANCASH,SANTA,4
5,CAJAMARCA,JAEN,6
6,CUSCO,LA CONVENCION,8
7,HUANUCO,HUANUCO,4
8,HUANUCO,PUERTO INCA,4
9,JUNIN,SATIPO,11


In [17]:
alt_worstProv=alt.Chart(worst_ProvYear_alarma_Frequency)

enc_worstProv=alt_worstProv.encode(
    y='departamento',
    x='provincia',
    text='yearsAffected:O',
    size='yearsAffected:O'
)

enc_worstProv.mark_text()

alt.Chart(...)

In [18]:
# --- Step 1: total cases per (year, departamento, enfermedad severity) ---
indexList = ['ano', 'departamento', 'enfermedad']
aggregator = {'case': ['sum']}
ByYearDepa = dengue.groupby(indexList, observed=True).agg(aggregator)

# --- Step 2: pivot enfermedad categories into columns ---
# fillna(0): a departamento-year missing a severity category entirely
# (e.g. no 2_ALARMA cases that year) becomes NaN after unstack; fill
# with 0 so the share computed below is 0, not NaN.
ByYearDepa_wide = ByYearDepa.unstack().fillna(0)

# --- Step 3: share of ALARMA cases out of all cases, per departamento-year ---
ByYearDepaAlarm = (
    ByYearDepa_wide.loc[:, ('case', 'sum', '2_ALARMA')] / ByYearDepa_wide.sum(axis=1)
)
ByYearDepaAlarm.name = 'alarmShare'
ByYearDepaAlarm = ByYearDepaAlarm.reset_index()

# --- Step 4: keep only departamento-years with some ALARMA share ---
# .copy() avoids SettingWithCopyWarning on the .loc assignment below --
# without it, ByYearDepaAlarm_focus is just a view into ByYearDepaAlarm,
# and pandas warns (and may eventually break) when you write into it.
ByYearDepaAlarm_focus = ByYearDepaAlarm[ByYearDepaAlarm.alarmShare > 0].copy()

# --- Step 5: bin the continuous share into labeled levels ---
# right-inclusive bins: (-1,.10], (.10,.25], (.25,.5], (.5,1]
# NOTE: a share of exactly 0.10 falls into "a.below10%" (it's really
# "at" 10%, not below) -- cosmetic edge case, doesn't affect the chart.
edges = [-1, .10, .25, .5, 1]
theLabels = ["a.below10%", "b.11-25%", "c.26-50%", "d.above50%"]
ByYearDepaAlarm_focus.loc[:, "alarmLevels"] = pd.cut(
    ByYearDepaAlarm_focus['alarmShare'],
    include_lowest=True,
    bins=edges,
    labels=theLabels,
    ordered=True,
)

# --- Step 6: rank departamentos by frequency of "high alarm" years ---
# "High alarm" = alarmLevels 'c.26-50%' or 'd.above50%' (share >= .5).
# This favors departamentos with recurring severe years over ones with
# a single extreme spike (unlike sorting by max or mean alarmShare).
# Adjust high_alarm_threshold if you want a stricter/looser cutoff.
high_alarm_threshold = 0.5

all_deptos = ByYearDepaAlarm_focus['departamento'].unique()

high_alarm_order = (
    ByYearDepaAlarm_focus[ByYearDepaAlarm_focus.alarmShare >= high_alarm_threshold]
    .groupby('departamento')['ano']
    .count()
    .reindex(all_deptos, fill_value=0)   # keep departamentos with 0 high-alarm years in the list
    .sort_values(ascending=False)
    .index.tolist()
)

ByYearDepaAlarm_focus

,ano,departamento,alarmShare,alarmLevels
33,2002,LORETO,0.000400,a.below10%
143,2010,JUNIN,0.021429,a.below10%
147,2010,LORETO,0.064297,a.below10%
148,2010,MADRE DE DIOS,0.004065,a.below10%
149,2010,PIURA,0.003336,a.below10%
...,...,...,...,...
365,2022,PASCO,0.161290,b.11-25%
366,2022,PIURA,0.117037,b.11-25%
368,2022,SAN MARTIN,0.215691,b.11-25%
369,2022,TUMBES,0.049793,a.below10%


Another plot

In [19]:
# --- Base chart: shared x/y across both layers ---
# y is sorted by high_alarm_order (frequency of severe years), computed
# in the wrangling step above -- NOT by a built-in EncodingSortField op,
# since Vega-Lite can't conditionally count "years above a threshold" on
# its own.
alt_WorstDepa = alt.Chart(ByYearDepaAlarm_focus).encode(
    x='ano:O',
    y=alt.Y('departamento:N', sort=high_alarm_order)
)

# --- Layer 1: heatmap coloring by binned alarm level ---
enc1_WorstDepa = alt_WorstDepa.encode(
    color=alt.Color('alarmLevels:O').scale(scheme="lightgreyred", reverse=False)
)

enc1_WorstDepa.mark_rect()

alt.Chart(...)

In [20]:
# --- Layer 2: numeric labels, shown only for cells >= 0.5 share ---
# threshold matches the color bin edge (Step 5: c.26-50% starts at .25),
# so every orange/red cell gets a number, not just a subset of them.
# format=".1f" kept as raw decimal (not %) per your call on chart density.
# fontWeight='bold' (not fontStyle) -- fontStyle only accepts normal/italic
# and silently fails to bold anything.
enc2_WorstDepa = alt_WorstDepa.encode(
    text=alt.Text('alarmShare:Q', format=".1f"),
    opacity=alt.condition('datum.alarmShare >= 0.5', alt.value(1), alt.value(0))
)

enc2_WorstDepa.mark_text(fontWeight='bold')

alt.Chart(...)

In [23]:
# @title
# --- Combine layers + title reflecting what the sort actually shows ---
final_chart = (enc1_WorstDepa.mark_rect() + enc2_WorstDepa.mark_text(fontWeight='bold')).properties(
    title=alt.TitleParams(
        text="Departamentos Ranked by Frequency of High-Alarm Years",
        subtitle="Sorted by count of years with alarma share >= 50% (recurrence, not peak intensity)",
    )
)

final_chart

alt.LayerChart(...)

You can find different color schemes [here](https://vega.github.io/vega/docs/schemes/)